# <center>**Titanic - Machine Learning from Disaster**<center>

## **1. Thư viện và cấu hình**

Cell này nạp các thư viện, cố định seed và chọn CPU/GPU. Kết quả là môi trường chạy nhất quán cho toàn bộ thí nghiệm.

In [1]:
import importlib
import random
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

try:
    optuna = importlib.import_module('optuna')
except ImportError:
    optuna = None

try:
    LGBMClassifier = importlib.import_module('lightgbm').LGBMClassifier
except ImportError:
    LGBMClassifier = None

SEED = 42
N_SPLITS = 5
N_TRIALS = 8
RUN_TUNING = False  # Đặt True để chạy Optuna trước khi huấn luyện ensemble.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = 'data'

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()
print(f'Using device: {DEVICE}')
print(f'LightGBM: {LGBMClassifier is not None}; Optuna: {optuna is not None}')

Using device: cpu
LightGBM: True; Optuna: True


## **2. Đọc dữ liệu và tạo đặc trưng**

Đọc `train.csv` và `test.csv`, sau đó tạo nhóm tuổi, họ, deck, mã vé, quy mô gia đình và tỷ lệ sống sót của nhóm đi cùng. Kết quả là hai bảng đặc trưng đồng nhất cho train/test.

In [2]:
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
test_df = pd.read_csv(f'{DATA_DIR}/test.csv')

def engineer_base(df):
    data = df.copy()
    data['Title'] = data['Name'].str.extract(r',\s*([^.]*)\.', expand=False).str.strip()
    rare_titles = ~data['Title'].isin(['Mr', 'Miss', 'Mrs', 'Master'])
    data.loc[rare_titles, 'Title'] = 'Rare'
    data['Title'] = data['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
    data['Surname'] = data['Name'].str.split(',').str[0].str.strip().str.lower()
    data['Deck'] = data['Cabin'].fillna('U').str[0].replace({'T': 'U'})
    data['TicketClean'] = data['Ticket'].fillna('UNKNOWN').str.upper().str.replace(r'[^A-Z0-9]', '', regex=True)
    data['TicketPrefix'] = data['Ticket'].fillna('UNKNOWN').str.upper().str.extract(r'^([A-Z]+)', expand=False).fillna('NONE')
    data['TicketLength'] = data['TicketClean'].str.len().astype(float)
    data['TicketFrequency'] = data['Ticket'].map(data['Ticket'].value_counts()).astype(float)
    data['FamilySize'] = data['SibSp'].fillna(0) + data['Parch'].fillna(0) + 1
    data['IsAlone'] = (data['FamilySize'] == 1).astype(int)
    data['FarePerPerson'] = data['Fare'].fillna(data['Fare'].median()) / data['TicketFrequency'].clip(lower=1)
    data['FamilyKey'] = data['Surname'] + '_' + data['TicketClean']
    return data

def add_family_survival_rate(data, reference=None, reference_y=None, leave_one_out=False, smoothing=3.0):
    result = data.copy()
    global_rate = float(np.mean(reference_y)) if reference_y is not None else 0.5
    if reference is None or reference_y is None:
        result['FamilySurvivalRate'] = global_rate
        return result
    ref = reference.reset_index(drop=True)
    labels = np.asarray(reference_y, dtype=float)
    result = result.reset_index(drop=True)
    rates = []
    for row_index, row in result.iterrows():
        candidates = [
            ('FamilyKey', row['FamilyKey']),
            ('TicketClean', row['TicketClean']),
            ('Surname', row['Surname']),
        ]
        value = global_rate
        for column, key in candidates:
            mask = ref[column].eq(key).to_numpy()
            count = int(mask.sum())
            total = float(labels[mask].sum())
            if leave_one_out and row_index < len(ref) and ref.loc[row_index, column] == key:
                count -= 1
                total -= labels[row_index]
            if count > 0:
                value = (total + smoothing * global_rate) / (count + smoothing)
                break
        rates.append(value)
    result['FamilySurvivalRate'] = rates
    return result

In [3]:
combined = engineer_base(pd.concat([train_df.drop(columns=['Survived']), test_df], ignore_index=True))

train_base = combined.iloc[:len(train_df)].reset_index(drop=True)
test_base = combined.iloc[len(train_df):].reset_index(drop=True)

y = train_df['Survived'].to_numpy(dtype=np.float32)

train_features = add_family_survival_rate(train_base, train_base, y, leave_one_out=True)
test_features = add_family_survival_rate(test_base, train_base, y, leave_one_out=False)

numeric_features = [
    'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone',
    'TicketLength', 'TicketFrequency', 'FarePerPerson', 'FamilySurvivalRate'
]
categorical_features = ['Sex', 'Embarked', 'Title', 'Deck', 'TicketPrefix']

print(f'Engineered train shape: {train_features.shape}; categorical: {categorical_features}')

Engineered train shape: (891, 23); categorical: ['Sex', 'Embarked', 'Title', 'Deck', 'TicketPrefix']


## **3. Mã hóa dữ liệu cho từng nhóm mô hình**

Biến số được điền khuyết và chuẩn hóa; biến phân loại được ánh xạ thành số nguyên cho Embedding. Đồng thời tạo ma trận one-hot gọn cho các mô hình cây.

In [4]:
# Biến số được chuẩn hóa; biến phân loại được mã hóa số nguyên cho nn.Embedding.
for column in numeric_features:
    median = train_features[column].median()
    train_features[column] = train_features[column].fillna(median).astype('float32')
    test_features[column] = test_features[column].fillna(median).astype('float32')

category_maps = {}
for column in categorical_features:
    values = pd.concat([train_features[column], test_features[column]], ignore_index=True).fillna('Unknown').astype(str)
    category_maps[column] = {value: index + 1 for index, value in enumerate(sorted(values.unique()))}
    train_features[column] = train_features[column].fillna('Unknown').astype(str).map(category_maps[column]).fillna(0).astype('int64')
    test_features[column] = test_features[column].fillna('Unknown').astype(str).map(category_maps[column]).fillna(0).astype('int64')

In [5]:
scaler = StandardScaler()
X_num = scaler.fit_transform(train_features[numeric_features]).astype('float32')
X_test_num = scaler.transform(test_features[numeric_features]).astype('float32')
X_cat = train_features[categorical_features].to_numpy(dtype=np.int64)
X_test_cat = test_features[categorical_features].to_numpy(dtype=np.int64)

# Tạo ma trận one-hot gọn cho các mô hình cây.
tree_train = pd.get_dummies(train_features[numeric_features + categorical_features], columns=categorical_features, dtype=float)
tree_test = pd.get_dummies(test_features[numeric_features + categorical_features], columns=categorical_features, dtype=float)
tree_test = tree_test.reindex(columns=tree_train.columns, fill_value=0)
X_tree = tree_train.to_numpy(dtype=np.float32)
X_test_tree = tree_test.to_numpy(dtype=np.float32)

print(f'PyTorch numeric/categorical shapes: {X_num.shape}, {X_cat.shape}; tree shape: {X_tree.shape}')

PyTorch numeric/categorical shapes: (891, 11), (891, 5); tree shape: (891, 49)


## **4. Mạng PyTorch với Embedding và Residual**

Mạng học Embedding cho biến phân loại, dùng các khối residual để truyền gradient ổn định và Focal Loss để tập trung vào mẫu khó. Kết quả là mô hình tabular có thể nhận đồng thời biến số và biến phân loại.

In [6]:
class TitanicDataset(Dataset):
    def __init__(self, numeric, categorical, labels=None):
        self.numeric = torch.tensor(numeric, dtype=torch.float32)
        self.categorical = torch.tensor(categorical, dtype=torch.long)
        self.labels = None if labels is None else torch.tensor(labels, dtype=torch.float32)
    def __len__(self):
        return len(self.numeric)
    def __getitem__(self, index):
        if self.labels is None:
            return self.numeric[index], self.categorical[index]
        return self.numeric[index], self.categorical[index], self.labels[index]

class ResidualBlock(nn.Module):
    def __init__(self, width, dropout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(width, width), nn.LayerNorm(width), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(width, width), nn.LayerNorm(width), nn.Dropout(dropout)
        )
    def forward(self, inputs):
        return torch.nn.functional.gelu(inputs + self.block(inputs))

class TitanicEmbeddingNet(nn.Module):
    def __init__(self, category_sizes, hidden=96, dropout=0.25, depth=2):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(size + 1, min(32, max(4, (size + 1) // 2)))
            for size in category_sizes
        ])
        embedding_width = sum(layer.embedding_dim for layer in self.embeddings)
        self.input = nn.Sequential(nn.Linear(X_num.shape[1] + embedding_width, hidden), nn.LayerNorm(hidden), nn.GELU())
        self.residuals = nn.Sequential(*[ResidualBlock(hidden, dropout) for _ in range(depth)])
        self.head = nn.Sequential(nn.Linear(hidden, max(16, hidden // 2)), nn.GELU(), nn.Dropout(dropout), nn.Linear(max(16, hidden // 2), 1))
    def forward(self, numeric, categorical):
        embedded = [layer(categorical[:, index]) for index, layer in enumerate(self.embeddings)]
        hidden = self.input(torch.cat([numeric] + embedded, dim=1))
        return self.head(self.residuals(hidden)).squeeze(1)

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.55, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probabilities = torch.sigmoid(logits)
        pt = probabilities * targets + (1 - probabilities) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (alpha_t * (1 - pt).pow(self.gamma) * bce).mean()

In [7]:
category_sizes = [len(category_maps[column]) for column in categorical_features]
model_probe = TitanicEmbeddingNet(category_sizes).to(DEVICE)

print(f'Embedding dimensions: {[layer.embedding_dim for layer in model_probe.embeddings]}')
print(f'Parameter count: {sum(parameter.numel() for parameter in model_probe.parameters()):,}')

Embedding dimensions: [4, 4, 4, 4, 11]
Parameter count: 46,991


## **5. Huấn luyện, tuning và stacking**

Định nghĩa vòng lặp PyTorch, các mô hình cây và Optuna. Sau đó huấn luyện theo 5 Stratified Folds, lấy xác suất OOF và kết hợp chúng bằng Logistic Regression.

In [8]:
def predict_torch(model, numeric, categorical):
    model.eval()
    loader = DataLoader(TitanicDataset(numeric, categorical), batch_size=256)
    predictions = []
    with torch.no_grad():
        for batch_num, batch_cat in loader:
            logits = model(batch_num.to(DEVICE), batch_cat.to(DEVICE))
            predictions.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(predictions)

def fit_torch(num_train, cat_train, labels, _num_valid, _cat_valid, params):
    _ = (_num_valid, _cat_valid)  # Giữ chữ ký chung cho các fold hiện tại.
    seed_everything()
    model = TitanicEmbeddingNet(
        category_sizes, params['hidden'], params['dropout'], params['depth']
    ).to(DEVICE)
    loader = DataLoader(
        TitanicDataset(num_train, cat_train, labels),
        batch_size=params.get('batch_size', 64), shuffle=True
    )
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=params['lr'], weight_decay=params['weight_decay']
    )
    criterion = FocalLoss(params.get('alpha', 0.55), params.get('gamma', 2.0))

    for _ in range(params.get('epochs', 90)):
        model.train()
        for batch_num, batch_cat, batch_y in loader:
            optimizer.zero_grad(set_to_none=True)
            logits = model(batch_num.to(DEVICE), batch_cat.to(DEVICE))
            loss = criterion(logits, batch_y.to(DEVICE))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            optimizer.step()
    return model

def make_xgb(params=None):
    config = {
        'n_estimators': 350, 'max_depth': 4, 'learning_rate': 0.035,
        'subsample': 0.85, 'colsample_bytree': 0.85,
        'min_child_weight': 2, 'reg_lambda': 2.0
    }
    config.update(params or {})
    return XGBClassifier(**config, random_state=SEED, eval_metric='logloss', n_jobs=-1)

def make_lgb(params=None):
    if LGBMClassifier is None:
        return None
    config = {
        'n_estimators': 300, 'num_leaves': 15, 'learning_rate': 0.03,
        'max_depth': 5, 'subsample': 0.85, 'colsample_bytree': 0.85,
        'reg_lambda': 2.0, 'verbosity': -1
    }
    config.update(params or {})
    return LGBMClassifier(**config, random_state=SEED, n_jobs=-1)

### **5.1. Hàm huấn luyện và mô hình nền**

Tạo các hàm dùng chung cho PyTorch, XGBoost và LightGBM. Kết quả là API thống nhất để huấn luyện và lấy xác suất.

In [9]:
def tune_models():
    if optuna is None:
        print('Không có Optuna, sử dụng cấu hình mặc định.')
        return {}, {}

    folds = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED)
    torch_space = {
        'hidden': 96, 'dropout': 0.25, 'depth': 2, 'lr': 8e-4,
        'weight_decay': 1e-3, 'epochs': 55, 'batch_size': 64
    }
    tree_spaces = {'xgb': {}, 'lgb': {}}

    def torch_objective(trial):
        params = {
            'hidden': trial.suggest_categorical('hidden', [64, 96, 128]),
            'dropout': trial.suggest_float('dropout', .1, .4),
            'depth': trial.suggest_int('depth', 1, 3),
            'lr': trial.suggest_float('lr', 2e-4, 3e-3, log=True),
            'weight_decay': trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True),
            'epochs': 45, 'batch_size': 64
        }
        scores = []
        for train_idx, valid_idx in folds.split(X_num, y):
            model = fit_torch(
                X_num[train_idx], X_cat[train_idx], y[train_idx],
                X_num[valid_idx], X_cat[valid_idx], params
            )
            prob = predict_torch(model, X_num[valid_idx], X_cat[valid_idx])
            scores.append(roc_auc_score(y[valid_idx], prob))
        return np.mean(scores)

    study = optuna.create_study(
        direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED)
    )
    study.optimize(torch_objective, n_trials=N_TRIALS, show_progress_bar=False)
    torch_space = study.best_params

    search_space = {
        'xgb': (make_xgb, {'max_depth': (3, 6), 'learning_rate': (.015, .08), 'min_child_weight': (1, 6)}),
        'lgb': (make_lgb, {'num_leaves': (8, 31), 'learning_rate': (.015, .08), 'max_depth': (3, 7)})
    }
    for name, (factory, space) in search_space.items():
        if factory() is None:
            continue

        def objective(trial, factory=factory, space=space):
            params = {
                key: trial.suggest_int(key, *bounds)
                if isinstance(bounds[0], int)
                else trial.suggest_float(key, *bounds)
                for key, bounds in space.items()
            }
            scores = []
            for train_idx, valid_idx in folds.split(X_tree, y):
                model = factory(params)
                model.fit(X_tree[train_idx], y[train_idx])
                prob = model.predict_proba(X_tree[valid_idx])[:, 1]
                scores.append(roc_auc_score(y[valid_idx], prob))
            return np.mean(scores)

        study = optuna.create_study(
            direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED)
        )
        study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
        tree_spaces[name] = study.best_params
    return torch_space, tree_spaces


torch_params, tree_params = tune_models() if RUN_TUNING else (
    {'hidden': 96, 'dropout': 0.25, 'depth': 2, 'lr': 8e-4,
     'weight_decay': 1e-3, 'epochs': 90, 'batch_size': 64},
    {'xgb': {}, 'lgb': {}}
)

print(f'Tuning: {RUN_TUNING}; số trial: {N_TRIALS}')

Tuning: False; số trial: 8


### **5.2. Tìm siêu tham số với Optuna**

Khi `RUN_TUNING = True`, Optuna tìm cấu hình tốt nhất bằng ROC-AUC trung bình trên 5 fold. Mặc định tắt để notebook chạy nhanh và tái lập kết quả cơ sở.

In [10]:
folds = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED)
oof = {name: np.zeros(len(y)) for name in ['PyTorch', 'XGBoost', 'LightGBM']}
test_predictions = {name: [] for name in oof}

for fold, (train_idx, valid_idx) in enumerate(folds.split(X_tree, y), 1):
    torch_model = fit_torch(
        X_num[train_idx], X_cat[train_idx], y[train_idx],
        X_num[valid_idx], X_cat[valid_idx], torch_params
    )
    oof['PyTorch'][valid_idx] = predict_torch(
        torch_model, X_num[valid_idx], X_cat[valid_idx]
    )
    test_predictions['PyTorch'].append(
        predict_torch(torch_model, X_test_num, X_test_cat)
    )

    for name, factory in [
        ('XGBoost', lambda: make_xgb(tree_params['xgb'])),
        ('LightGBM', lambda: make_lgb(tree_params['lgb']))
    ]:
        model = factory()
        model.fit(X_tree[train_idx], y[train_idx])
        oof[name][valid_idx] = model.predict_proba(X_tree[valid_idx])[:, 1]
        test_predictions[name].append(model.predict_proba(X_test_tree)[:, 1])
    print(f'Fold {fold}/{N_SPLITS} hoàn tất')

for name in test_predictions:
    test_predictions[name] = np.mean(test_predictions[name], axis=0)

meta_features = pd.DataFrame(oof)
meta_model = LogisticRegression(C=0.5, max_iter=1000, random_state=SEED)
meta_model.fit(meta_features, y)
stacked_oof = meta_model.predict_proba(meta_features)[:, 1]

print(f'Stacking ROC-AUC: {roc_auc_score(y, stacked_oof):.4f}')

Fold 1/5 hoàn tất
Fold 2/5 hoàn tất
Fold 3/5 hoàn tất
Fold 4/5 hoàn tất
Fold 5/5 hoàn tất
Stacking ROC-AUC: 0.8962


## **6. Đánh giá mô hình**

So sánh Accuracy, F1-score và ROC-AUC trên xác suất OOF. Kết quả phản ánh khả năng tổng quát hóa qua đủ 5 fold thay vì chỉ một lần chia dữ liệu.

In [11]:
evaluation_rows = []

for name, probabilities in oof.items():
    predictions = (probabilities >= 0.5).astype(int)
    evaluation_rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y, predictions),
        'F1-Score': f1_score(y, predictions),
        'ROC-AUC': roc_auc_score(y, probabilities),
    })
evaluation_rows.append({
    'Model': 'Stacking Logistic Regression',
    'Accuracy': accuracy_score(y, stacked_oof >= 0.5),
    'F1-Score': f1_score(y, stacked_oof >= 0.5),
    'ROC-AUC': roc_auc_score(y, stacked_oof),
})

results_df = pd.DataFrame(evaluation_rows).sort_values('ROC-AUC', ascending=False)
print(results_df.to_string(index=False))

                       Model  Accuracy  F1-Score  ROC-AUC
                     XGBoost  0.854097  0.801829 0.900398
Stacking Logistic Regression  0.850730  0.795699 0.896207
                    LightGBM  0.842873  0.785276 0.894545
                     PyTorch  0.827160  0.768769 0.850211


## **7. Tạo file submission**

Dùng meta-learner để suy ra xác suất trên test và xuất hai cột `PassengerId`, `Survived`. Kết quả là file `submission.csv` sẵn sàng nộp Kaggle.

In [12]:
test_meta_features = pd.DataFrame(test_predictions).reindex(columns=meta_features.columns)

final_probabilities = meta_model.predict_proba(test_meta_features)[:, 1]
final_predictions = (final_probabilities >= 0.5).astype(int)
submission = pd.DataFrame({'PassengerId': test_df['PassengerId'], 'Survived': final_predictions})
submission.to_csv(f'{DATA_DIR}/submission.csv', index=False)

print(f'Saved {DATA_DIR}/submission.csv with {len(submission):,} predictions.')
print(submission['Survived'].value_counts(normalize=True).rename('share').to_string())

Saved data/submission.csv with 418 predictions.
Survived
0    0.626794
1    0.373206


## **8. Tổng kết kết quả đạt được trên Kaggle**

Mô hình đạt **Accuracy = 0.77511** trên tập kiểm tra ẩn của Kaggle. Tập test có 418 hành khách, vì vậy mô hình dự đoán đúng khoảng **324 hành khách**.

Accuracy được tính theo công thức:

$$
\text{Accuracy} = \frac{\text{Số dự đoán đúng}}{\text{Tổng số mẫu kiểm tra}}
$$

### **Ý nghĩa của kết quả**

Kết quả cho thấy mô hình đã học được những yếu tố quan trọng ảnh hưởng đến khả năng sống sót, như giới tính, hạng vé, tuổi, giá vé, quy mô gia đình, deck và nhóm hành khách dùng chung vé.

Điểm Kaggle thấp hơn Accuracy khoảng **0.8507** trên Cross Validation nội bộ. Điều này xảy ra vì Cross Validation được đánh giá trên các phần dữ liệu tách từ tập train, còn Kaggle đánh giá trên tập test hoàn toàn ẩn nhãn. Ngoài ra, bộ dữ liệu Titanic khá nhỏ nên kết quả có thể thay đổi đáng kể chỉ với một số dự đoán sai.

### **Đánh giá cuối cùng**

Mức độ chính xác **77.511%** cho thấy pipeline đã hoạt động tốt trên dữ liệu chưa nhìn thấy, nhưng vẫn còn khoảng cách giữa kết quả validation và kết quả thực tế trên Kaggle. Đây là dấu hiệu mô hình cần được cải thiện thêm về khả năng tổng quát hóa, lựa chọn đặc trưng và ngưỡng phân loại.

> **Kết luận:** Mô hình đạt kết quả thực tế là **77.511% chính xác trên Kaggle**. Kết quả này phản ánh chính xác hơn khả năng dự đoán trên dữ liệu mới so với chỉ dựa vào điểm validation nội bộ.

# <center>**HẾT**<center>